# Tutorial 1 - Simple examples

In [ ]:
from tikzpics import TikzFigure

%load_ext autoreload
%autoreload 2

In [ ]:
calendar_year = 2026
calendar_width = 75
calendar_height = 50
num_months = 12
month_width = calendar_width / num_months
day_height = calendar_height / 31
padding = 0.05

In [ ]:
import calendar
import datetime

import requests

# Check if it's a national holiday in Sweden
# curl -X GET 'https://openholidaysapi.org/PublicHolidays?countryIsoCode=CH&languageIsoCode=DE&validFrom=2022-01-01&validTo=2022-06-30' -H 'accept: text/json' | ConvertFrom-Json | ConvertTo-Json


def get_holidays(country_code, year):
    response = requests.get(
        f"https://openholidaysapi.org/PublicHolidays?countryIsoCode={country_code}&validFrom={year}-01-01&validTo={year}-12-31"
    )
    holidays = response.json()
    return holidays


holidays_dict = {
    "SE": get_holidays("SE", calendar_year),
    "DE": get_holidays("DE", calendar_year),
    "FR": get_holidays("FR", calendar_year),
    "LB": get_holidays("LB", calendar_year),
}

In [ ]:
def date_to_index(year, month, day):
    return datetime.date(year, month, day).timetuple().tm_yday - 1


dates = []
for month in range(1, num_months + 1):
    num_days = calendar.monthrange(calendar_year, month)[1]
    for day in range(1, num_days + 1):
        date = datetime.date(calendar_year, month, day)
        date_dict = {
            "date": date,
            "is_sunday": date.weekday() == 6,
            "is_saturday": date.weekday() == 5,
            "holiday_se": None,
            "holiday_de": None,
            "holiday_fr": None,
            "holiday_lb": None,
        }
        dates.append(date_dict)

for country_code, holidays in holidays_dict.items():
    for holiday in holidays:
        calendar_year, month, day = map(int, holiday["startDate"].split("-"))
        index = date_to_index(calendar_year, month, day)
        dates[index][f"holiday_{country_code.lower()}"] = holiday["name"][0]["text"]

In [ ]:
fig = TikzFigure()

nodes = [
    (0, 0),
    (calendar_width, 0),
    (calendar_width, calendar_height),
    (0, calendar_height),
]
fig.draw(nodes=nodes, fill="gray!10!white", draw="black", cycle=True)


def get_color(month: int, day: int) -> str:
    """Returns the color for a given month and day.

    if sunday, return red
    if saturday, return red
    if weekday, return white
    """

    index = date_to_index(calendar_year, month + 1, day + 1)
    date = dates[index]

    if date["holiday_se"] is not None:
        return "orange!95!white"
    if date["holiday_de"] is not None:
        return "orange!95!white"
    if date["holiday_fr"] is not None:
        return "orange!95!white"
    if date["holiday_lb"] is not None:
        return "orange!95!white"

    if date["is_sunday"]:  # Sunday
        return "red!20!white"
    elif date["is_saturday"]:  # Saturday
        return "red!10!white"
    else:  # Weekday
        return "white"


for month in range(num_months):
    x_center = month * month_width + month_width / 2
    month_name = calendar.month_name[month + 1]
    fig.add_node(
        x=x_center,
        y=calendar_height * 1.02,
        anchor="center",
        scale=3.0,
        content=month_name,
    )
    # Get number of days in the month
    num_days = calendar.monthrange(calendar_year, month + 1)[1]
    for day in range(num_days):
        y_center = calendar_height - (day * day_height + day_height / 2)

        # print(f"  Day {day + 1}: y = {y:.2f}")
        x0 = x_center - month_width / 2 + padding
        y0 = y_center - day_height / 2 + padding

        nodes = [
            (x0, y0),
            (x0 + month_width - 2 * padding, y0),
            (x0 + month_width - 2 * padding, y0 + day_height - 2 * padding),
            (x0, y0 + day_height - 2 * padding),
        ]
        col = get_color(month, day)
        fig.draw(nodes=nodes, fill=col, draw=col, cycle=True)
        fig.add_node(x=x0, y=y_center, anchor="west", scale=2.0, content=f"{day + 1}")

fig.show()

In [ ]:
print(fig.generate_standalone())